In [19]:
import os
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

# 1. FIX: Load environment variables from .env file
load_dotenv()

hug_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hug_token,
    max_new_tokens=300
)

model = ChatHuggingFace(llm=llm)

# 2. Pydantic Schema for Evaluation
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation")
    feedback: str = Field(..., description="Feedback for the tweet.")

parser = PydanticOutputParser(pydantic_object=TweetEvaluation)

# 3. FIX: State Definition (Fixed "needs_improvement" spelling)
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

# 4. Node Functions
def generate_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]
    tweet = model.invoke(messages).content
    return {"tweet": tweet}

def evaluate_tweet(state: TweetState):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
        ("human", """Evaluate the following tweet:

Tweet: "{tweet}"

Use the criteria below to evaluate the tweet:
1. Originality - Is this fresh, or have you seen it a hundred times before?
2. Humor - Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness - Is it short, sharp, and scroll-stopping?
4. Virality Potential - Would people retweet or share it?
5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke

{format_instructions}""")
    ])

    chain = prompt | model | parser
    response = chain.invoke({
        "tweet": state["tweet"],
        "format_instructions": parser.get_format_instructions()
    })

    return {"evaluation": response.evaluation, "feedback": response.feedback}

def optimize_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = model.invoke(messages).content
    current_iteration = state.get('iteration', 1)

    return {'tweet': response, 'iteration': current_iteration + 1}

# 5. Conditional Router
def route_evaluation(state: TweetState):
    if state.get('evaluation') == 'approved' or state.get('iteration', 1) >= state.get('max_iteration', 5):
        return 'approved'
    else:
        return 'needs_improvement'

# 6. Build Graph
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')

graph.add_conditional_edges(
    'evaluate',
    route_evaluation,
    {'approved': END, 'needs_improvement': 'optimize'}
)
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()

# 7. Execution
initial_state = {
    "topic": "Swat Pakistan Pakhtun Khwa",
    "iteration": 1,
    "max_iteration": 5
}

result = workflow.invoke(initial_state)

print("\n=== FINAL GENERATED TWEET ===")
print("Tweet:", result.get("tweet"))
print("Evaluation:", result.get("evaluation"))
print("Iterations:", result.get("iteration"))
print("Feedback:", result.get("feedback"))


=== FINAL GENERATED TWEET ===
Tweet: Just ordered Swat Pakhtun Khwa from UberEats. Delivery time: 12 hours. Plus, they forgot to include a bagpipe performance. #SwatPakhtunKhwa #PatienceIsAGift
Evaluation: approved
Iterations: 1
Feedback: The tweet is original, playing on the long delivery time and the unexpected service included in the Swat Pakhtun Khwa order. It has a nice touch of dry humor that would likely resonate with viewers. The tweet is punchy, concise, and within the character limit. It also carries a viral potential due to its relatable and humorous premise.
